In [1]:
import pandas as pd
import numpy as np
import json
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


StatsBomb Performance

In [2]:
events_root = Path("../data/raw/statsbomb/events")

event_files = list(events_root.glob("*.json"))[:600]
events_list = []

for file in event_files:
    with open(file, encoding="utf-8") as f:
        data = json.load(f)
        df = pd.json_normalize(data)
        events_list.append(df)

events_df = pd.concat(events_list, ignore_index=True)

print("Total events:", events_df.shape)


Total events: (2115619, 149)


In [3]:
# shots
shots = events_df[events_df['type.name'] == 'Shot']
shots_per_player = shots.groupby('player.name').size().reset_index(name='shots')

# goals
goals = events_df[
    (events_df['type.name'] == 'Shot') &
    (events_df['shot.outcome.name'] == 'Goal')
]
goals_per_player = goals.groupby('player.name').size().reset_index(name='goals')

# passes
passes = events_df[events_df['type.name'] == 'Pass']
passes_per_player = passes.groupby('player.name').size().reset_index(name='passes')

# minutes proxy
minutes_per_player = events_df.groupby('player.name').size().reset_index(name='total_minutes')


In [4]:
performance_df = shots_per_player.merge(goals_per_player, on='player.name', how='left')
performance_df = performance_df.merge(passes_per_player, on='player.name', how='left')
performance_df = performance_df.merge(minutes_per_player, on='player.name', how='left')

performance_df = performance_df.fillna(0)


In [5]:
performance_df.rename(columns={'player.name': 'player_name'}, inplace=True)

performance_df['player_name'] = (
    performance_df['player_name']
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)


In [6]:
performance_df.to_csv("../data/processed/performance_data.csv", index=False)


In [18]:
filtered_perf = performance_df[performance_df['total_minutes'] >= 900]


In [19]:
filtered_perf = performance_df.head(1000)


Transfermarkt Scraping

In [21]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re

# ---------- parser ----------
def parse_market_value(value):
    if pd.isna(value):
        return np.nan

    value = str(value).lower().replace('€', '').strip()

    multiplier = 1
    if 'm' in value:
        multiplier = 1
    elif 'k' in value:
        multiplier = 0.001

    num = re.findall(r'[\d,.]+', value)
    if not num:
        return np.nan

    num = float(num[0].replace(',', ''))
    return num * multiplier


# ---------- driver ----------
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

driver.get("https://www.transfermarkt.co.in/")
time.sleep(4)

# ---------- player universe ----------
player_list = (
    filtered_perf['player_name']
    .dropna()
    .str.strip()
    .unique()
    .tolist()
)

print("Players to search:", len(player_list))

results = []

# ---------- MAIN LOOP ----------
for i, player in enumerate(player_list):
    try:
        print(f"[{i+1}/{len(player_list)}] Searching:", player)

        # find search box fresh every time
        search_box = driver.find_element(By.NAME, "query")
        search_box.clear()
        search_box.send_keys(player)
        search_box.send_keys(Keys.RETURN)

        time.sleep(3)

        # open first result
        first_result = driver.find_element(
            By.CSS_SELECTOR,
            "table.items tbody tr td.hauptlink a"
        )
        first_result.click()

        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        value_tag = soup.find("a", class_="data-header__market-value-wrapper")

        raw_value = value_tag.text.strip() if value_tag else np.nan

        results.append({
            "player_name": player,
            "market_value_raw": raw_value
        })

    except Exception as e:
        print("Failed:", player)

        results.append({
            "player_name": player,
            "market_value_raw": np.nan
        })

    # 🔥 IMPORTANT — always go home
    driver.get("https://www.transfermarkt.co.in/")
    time.sleep(2)

driver.quit()

# ---------- dataframe ----------
market_df = pd.DataFrame(results)

market_df['market_value'] = market_df['market_value_raw'].apply(parse_market_value)

market_df['player_name'] = (
    market_df['player_name']
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

market_df = market_df[['player_name', 'market_value']]

print(market_df.head())
print("Rows:", market_df.shape[0])

# ---------- save ----------
market_df.to_csv("../data/processed/market_value_data.csv", index=False)

Players to search: 1000
[1/1000] Searching: aaron cresswell
[2/1000] Searching: aaron lennon
[3/1000] Searching: aaron ramsey
[4/1000] Searching: abbey-leigh stringer
Failed: abbey-leigh stringer
[5/1000] Searching: abbi grant
[6/1000] Searching: abbie mcmanus
Failed: abbie mcmanus
[7/1000] Searching: abby erceg
Failed: abby erceg
[8/1000] Searching: abdoul karim yoda
[9/1000] Searching: abdoulaye ba
[10/1000] Searching: abdoulaye doucouré
[11/1000] Searching: abdul rahman baba
[12/1000] Searching: abdón prats bastidas
[13/1000] Searching: abel eduardo balbo
[14/1000] Searching: abel ruiz ortega
[15/1000] Searching: abigail harrison
[16/1000] Searching: adam david lallana
[17/1000] Searching: adam johnson
[18/1000] Searching: adam smith
[19/1000] Searching: adelina engman
Failed: adelina engman
[20/1000] Searching: adlène guédioura
[21/1000] Searching: adnan januzaj
[22/1000] Searching: adrian mutu
[23/1000] Searching: adriana kristina leon
[24/1000] Searching: adriano correia claro
[2

In [22]:
# save clean (production)
clean_df = market_df.dropna(subset=['market_value'])
clean_df.to_csv("../data/processed/market_value_data.csv", index=False)

In [23]:
overlap = len(
    set(performance_df.player_name) &
    set(market_df.player_name)
)

print("PERF ∩ MARKET =", overlap)


PERF ∩ MARKET = 1000


In [24]:
print("Performance players:", performance_df.shape[0])
print("Market players:", market_df.shape[0])
print("Overlap:", overlap)


Performance players: 1759
Market players: 1000
Overlap: 1000


In [25]:
print("Market value coverage:",
      market_df['market_value'].notna().mean())

print("Players with value:",
      market_df['market_value'].notna().sum())


Market value coverage: 0.268
Players with value: 268


SENTIMENT PROCESSING

In [2]:
import pandas as pd
import numpy as np

# load market players
market_df = pd.read_csv("../data/processed/market_value_data.csv")

# 🔥 USE THIS FILE (your best one)
sentiment_raw = pd.read_csv(
    "../data/raw/sentiment-dataset.csv",
    encoding="latin-1"
)

print("Sentiment raw:", sentiment_raw.shape)

Sentiment raw: (167841, 9)


In [3]:
sentiment_raw['player_name'] = (
    sentiment_raw['player_name']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

In [4]:
valid_players = market_df.loc[
    market_df['market_value'].notna(),
    'player_name'
].unique().tolist()

print("Players with market value:", len(valid_players))

Players with market value: 268


In [5]:
sentiment_filtered = sentiment_raw[
    sentiment_raw['player_name'].isin(valid_players)
].copy()

print("Matched sentiment rows:", sentiment_filtered.shape)

Matched sentiment rows: (14086, 9)


In [6]:
sentiment_df = (
    sentiment_filtered
    .groupby('player_name')
    .agg({
        'vader_polarity': 'mean'
    })
    .reset_index()
    .rename(columns={'vader_polarity': 'sentiment_score'})
)

print("Player sentiment rows:", sentiment_df.shape)

Player sentiment rows: (32, 2)


In [7]:
sentiment_df = market_df[['player_name']].merge(
    sentiment_df,
    on='player_name',
    how='left'
)

sentiment_df['sentiment_score'] = sentiment_df['sentiment_score'].fillna(0)

In [8]:
def label_sentiment(score):
    if score > 0.05:
        return "positive"
    elif score < -0.05:
        return "negative"
    else:
        return "neutral"

sentiment_df['sentiment_label'] = sentiment_df['sentiment_score'].apply(label_sentiment)

In [9]:
sentiment_df.to_csv(
    "../data/processed/sentiment_data.csv",
    index=False
)

print("✅ Sentiment data saved:", sentiment_df.shape)

✅ Sentiment data saved: (268, 3)


In [10]:
print(sentiment_df['sentiment_score'].describe())
print("Unique players with sentiment:",
      (sentiment_df['sentiment_score'] != 0).sum())

count    268.000000
mean       0.018560
std        0.055574
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.403474
Name: sentiment_score, dtype: float64
Unique players with sentiment: 32


INJURY DATA

In [8]:
import pandas as pd
import numpy as np
import re

injury_raw = pd.read_csv("../data/raw/injury_data.csv")

print("Columns:", injury_raw.columns.tolist())
print("Raw injury rows:", injury_raw.shape)



Columns: ['Season', 'Injury', 'Days', 'Games missed', 'injury_from_parsed', 'injury_until_parsed', 'player_name', 'player_age', 'player_position', 'club', 'league']
Raw injury rows: (15603, 11)


In [9]:
injury_raw['player_name'] = (
    injury_raw['player_name']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)


In [10]:
import re
import numpy as np

def extract_days(text):
    if pd.isna(text):
        return 0
    nums = re.findall(r'(\d+)', str(text))
    return int(nums[0]) if nums else 0

injury_raw['total_days_out'] = injury_raw['Days'].apply(extract_days)


In [11]:
injury_raw['games_missed'] = pd.to_numeric(
    injury_raw['Games missed'],
    errors='coerce'
).fillna(0)


In [12]:
injury_df = (
    injury_raw
    .groupby('player_name')
    .agg({
        'total_days_out': 'sum',
        'games_missed': 'sum'
    })
    .reset_index()
)

print("Aggregated injury players:", injury_df.shape)


Aggregated injury players: (4081, 3)


In [15]:
import pandas as pd

# reload only what you need (fast)
market_df = pd.read_csv("../data/processed/market_value_data.csv")

valid_players = market_df.loc[
    market_df['market_value'].notna(),
    'player_name'
].unique().tolist()

injury_df = injury_df[
    injury_df['player_name'].isin(valid_players)
].copy()

In [16]:
base_players = pd.DataFrame({'player_name': valid_players})

injury_df = base_players.merge(
    injury_df,
    on='player_name',
    how='left'
)

injury_df['total_days_out'] = injury_df['total_days_out'].fillna(0)
injury_df['games_missed'] = injury_df['games_missed'].fillna(0)

print("Final injury players:", injury_df.shape)


Final injury players: (268, 3)


In [17]:
injury_df.to_csv("../data/processed/injury_data.csv", index=False)


In [20]:
import pandas as pd

# reload missing dataframe
sentiment_df = pd.read_csv("../data/processed/sentiment_data.csv")

print("Performance players:", performance_df.shape[0])
print("Market players:", market_df.shape[0])
print("Sentiment players:", sentiment_df.shape[0])
print("Injury players:", injury_df.shape[0])

Performance players: 1759
Market players: 268
Sentiment players: 268
Injury players: 268


In [21]:
import pandas as pd
import numpy as np

performance_df = pd.read_csv("../data/processed/performance_data.csv")
market_df = pd.read_csv("../data/processed/market_value_data.csv")
sentiment_df = pd.read_csv("../data/processed/sentiment_data.csv")
injury_df = pd.read_csv("../data/processed/injury_data.csv")

print("Loaded.")


Loaded.


In [24]:
final_df = market_df.copy()

final_df = final_df.merge(
    performance_df,
    on="player_name",
    how="left"
)

final_df = final_df.merge(
    sentiment_df,
    on="player_name",
    how="left"
)

final_df = final_df.merge(
    injury_df,
    on="player_name",
    how="left"
)

print("Final shape:", final_df.shape)


Final shape: (268, 10)


In [25]:
print(final_df.isnull().mean().sort_values(ascending=False).head(10))


player_name        0.0
market_value       0.0
shots              0.0
goals              0.0
passes             0.0
total_minutes      0.0
sentiment_score    0.0
sentiment_label    0.0
total_days_out     0.0
games_missed       0.0
dtype: float64


In [26]:
final_df.drop_duplicates(inplace=True)


In [27]:
numeric_cols = final_df.select_dtypes(include="number").columns.tolist()
numeric_cols.remove("market_value")  # never touch target

final_df[numeric_cols] = final_df[numeric_cols].fillna(
    final_df[numeric_cols].median()
)


In [28]:
cat_cols = final_df.select_dtypes(include="object").columns.tolist()
cat_cols.remove("player_name")

for col in cat_cols:
    final_df[col] = final_df[col].fillna(
        final_df[col].mode()[0] if not final_df[col].mode().empty else "unknown"
    )


C:\Users\abhis\AppData\Local\Temp\ipykernel_20688\3774255614.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = final_df.select_dtypes(include="object").columns.tolist()


In [29]:
final_df["goal_conversion_rate"] = (
    final_df["goals"] / final_df["shots"].replace(0, np.nan)
)

final_df["goal_conversion_rate"] = final_df["goal_conversion_rate"].fillna(0)


In [30]:
final_df["minutes_per_goal"] = (
    final_df["total_minutes"] / final_df["goals"].replace(0, np.nan)
)

final_df["minutes_per_goal"] = final_df["minutes_per_goal"].fillna(
    final_df["total_minutes"]
)


In [31]:
final_df["performance_index"] = (
    0.4 * final_df["goals"]
    + 0.3 * final_df["shots"]
    + 0.3 * final_df["passes"]
)


In [36]:
final_df["injury_risk_score"] = (
    final_df["total_days_out"] / (final_df["total_minutes"] + 1)
)


In [37]:
final_df = pd.get_dummies(
    final_df,
    columns=["sentiment_label"],
    drop_first=True
)


KeyError: "None of [Index(['sentiment_label'], dtype='str')] are in the [columns]"

In [47]:
final_df.to_csv("../data/processed/final_clean_dataset.csv", index=False)


In [48]:
X = final_df.drop(columns=["player_name", "market_value"])
y = final_df["market_value"]

X.to_csv("../data/processed/model_features_scaled.csv", index=False)
y.to_csv("../data/processed/model_target.csv", index=False)

print("Week-2 artifacts saved.")


Week-2 artifacts saved.


In [49]:
print(final_df.shape)
print(final_df.isnull().sum().sort_values(ascending=False).head(10))
print(final_df.head())


(91, 13)
player_name             0
market_value            0
shots                   0
goals                   0
passes                  0
total_minutes           0
sentiment_score         0
total_days_out          0
games_missed            0
goal_conversion_rate    0
dtype: int64
                 player_name  market_value     shots     goals    passes  \
0           abel ruiz ortega          24.0 -0.516484 -0.418004 -0.373444   
1         adrián marín gómez          12.0 -0.516484 -0.418004 -0.396822   
2       aleix garcía serrano         160.0 -0.516484 -0.418004 -0.181738   
3  alejandro gallar falguera           1.2 -0.516484  0.809040 -0.340713   
4    alejandro menéndez díez          40.0 -0.274216 -0.418004 -0.340713   

   total_minutes  sentiment_score  total_days_out  games_missed  \
0      -0.400581    -2.775558e-17       -0.328981     -0.357018   
1      -0.442576    -2.775558e-17       -0.328981     -0.357018   
2      -0.217700    -2.775558e-17       -0.328981     -0.357

In [50]:
print(X.shape)
print(y.shape)


(91, 11)
(91,)
